# Project D, Person 2: Baseline and DCF

This notebook is my part of Project D. It builds the three-year-average baseline that the
language model has to beat, a DCF that turns any growth and margin forecast into an
enterprise value, and the sensitivity tables.

The DCF gets tested on made-up numbers first (Part 2) and only then run on the 53 firms
(Part 3). When Person 3's forecasts are ready they go in at Part 3, and everything after
that updates on its own.

## Part 0: Data and assumptions

In [1]:
import os
import numpy as np
import pandas as pd

DATA = os.path.join("..", "data", "master.csv")
OUT = "output"
os.makedirs(OUT, exist_ok=True)

master = pd.read_csv(DATA)
mx = master.set_index("ticker")

print(f"firms          : {len(master)}")
print(f"target year    : {master['sample_year'].unique()}")
print(f"missing values : {master.isna().sum().sum()}")
print(f"duplicated     : {master['ticker'].duplicated().sum()}")
print(f"backlog firms  : {master['backlog_mentioned'].sum()} of {len(master)}")

firms          : 53
target year    : [2025]
missing values : 0
duplicated     : 0
backlog firms  : 38 of 53


Four firms have histories that don't line up year to year. We kept them in, but every
summary table also shows the results without them.

In [2]:
FLAGGED = {
    "DD":  "FY22 revenue on a different basis to FY23-24 (restated); 'margin' is pre-tax",
    "FTV": "FY22 revenue on a different basis to FY23-24 (spin-off restatement)",
    "MMM": "FY23 margin of -43%: one-off litigation charges",
    "BA":  "operating losses FY22-24, then +34% revenue in FY25",
}
mx["flag"] = pd.Series(FLAGGED).reindex(mx.index).fillna("")

print(mx.loc[mx["flag"] != "", ["flag"]].to_string())

                                                                                flag
ticker                                                                              
MMM                                  FY23 margin of -43%: one-off litigation charges
BA                               operating losses FY22-24, then +34% revenue in FY25
DD      FY22 revenue on a different basis to FY23-24 (restated); 'margin' is pre-tax
FTV              FY22 revenue on a different basis to FY23-24 (spin-off restatement)


### Assumptions

| Input | Value | Type | Source |
| :--- | :--- | :--- | :--- |
| FY2024 revenue | per firm | filing | `revenue_fy_minus_1` (XBRL) |
| Year-1 growth and margin | per firm | the forecast being tested | baseline, last year, AI, or actual |
| Risk-free rate | 4.5% | assumed | 10-yr Treasury closed at 4.49% on 7 Feb 2025, when the FY2024 10-Ks came out |
| Equity risk premium | 5.0% | assumed | same ERP as Lecture 7. Damodaran's implied ERP was 4.33% on 1 Jan 2025, so we run that as a sensitivity |
| Beta | 1.00 | assumed | Damodaran industry betas: Machinery 0.96, Aerospace/Defense 0.95 |
| Debt / capital | 13% | assumed | Damodaran: Machinery 12.8%, Aerospace/Defense 13.5% |
| After-tax cost of debt | 4.0% | assumed | Damodaran, Machinery: 5.29% pre-tax, 3.97% after tax |
| WACC | 8.8% | computed | from the five rows above |
| Terminal growth | 2.5% | assumed | below the risk-free rate, and under the ~3% Lecture 7 warns about |
| Tax rate | 25% | assumed | 21% federal plus state. Damodaran's 5.29% to 3.97% implies the same 25% |
| ROIC | 20% | assumed | Damodaran after-tax ROIC: Machinery 24.4%, Aerospace/Defense 16.0%, Electrical Equipment 15.6% |
| Horizon | 5 years | assumed | FY2025 to FY2029, same as Lectures 7 and 8 |

The Damodaran numbers are from his January 2026 tables, while our valuation date is
February 2025. Industry betas and debt levels don't move much in a year, so we didn't go back
for the older tables.

These are the same for every forecast. They change how big the values are, but the only
thing that differs between scenarios is year-1 growth and margin. Part 4 checks whether
they could change which forecast comes out ahead.

In [3]:
# discount rate, same CAPM setup as Lecture 7
RF, ERP, BETA = 0.045, 0.050, 1.00   # 10-yr UST (Feb 2025), course ERP, industry beta
KD_AT, D_WT = 0.040, 0.13            # after-tax cost of debt, debt weight

ke = RF + BETA * ERP                 # cost of equity
WACC = round((1 - D_WT) * ke + D_WT * KD_AT, 3)

# everything else
G_TERM, YEARS = 0.025, 5   # terminal growth, forecast years (FY25-FY29)
TAX, ROIC = 0.25, 0.20     # tax rate, reinvestment = growth / ROIC

print(f"cost of equity {ke:.2%}  at {1 - D_WT:.0%} of capital")
print(f"cost of debt   {KD_AT:.2%}  at {D_WT:.0%} of capital (after tax)")
print(f"WACC           {WACC:.2%}")
print(f"\nimplied exit multiple of FCF: {(1 + G_TERM) / (WACC - G_TERM):.1f}x")

cost of equity 9.50%  at 87% of capital
cost of debt   4.00%  at 13% of capital (after tax)
WACC           8.80%

implied exit multiple of FCF: 16.3x


## Part 1: Baseline

The baseline is the simple rule from the project description: assume next year looks like
the average of the last three.

Margin is a straight average of FY2022, FY2023 and FY2024. Growth is the average of FY2023
and FY2024 growth, since three years of revenue only give two growth rates (the data README
leaves FY2021 out on purpose to stay clear of the pandemic rebound).

We also added a second benchmark that just uses FY2024 growth and margin. It picks up
momentum, so if the model beats the three-year average but loses to last year, it might be
reading momentum and not backlog.

In [4]:
def baseline(row):
    """3-yr average forecast for one firm -> (growth, margin)"""
    g23 = row["revenue_fy_minus_2"] / row["revenue_fy_minus_3"] - 1
    g24 = row["revenue_fy_minus_1"] / row["revenue_fy_minus_2"] - 1
    growth = (g23 + g24) / 2          # only 2 growth rates in FY22-FY24
    margin = (row["margin_fy_minus_3"] + row["margin_fy_minus_2"]
              + row["margin_fy_minus_1"]) / 3
    return growth, margin


def last_year(row):
    """FY2024 only -> (growth, margin)"""
    growth = row["revenue_fy_minus_1"] / row["revenue_fy_minus_2"] - 1
    return growth, row["margin_fy_minus_1"]


rows = []
for tk, r in mx.iterrows():
    bg, bm = baseline(r)
    lg, lm = last_year(r)
    rows.append({"ticker": tk,
                 "base_growth": bg, "base_margin": bm,
                 "last_growth": lg, "last_margin": lm,
                 "actual_growth": r["actual_revenue_growth"],
                 "actual_margin": r["actual_margin"],
                 "backlog": bool(r["backlog_mentioned"]),
                 "flag": r["flag"]})
fc = pd.DataFrame(rows).set_index("ticker")

print(fc.drop(columns="flag").head(6).round(4).to_string())

        base_growth  base_margin  last_growth  last_margin  actual_growth  actual_margin  backlog
ticker                                                                                           
MMM         -0.0304      -0.0237      -0.0014       0.1962         0.0152         0.1855    False
ALLE         0.0745       0.1934       0.0333       0.2070         0.0782         0.2113    False
AME          0.0624       0.2531       0.0522       0.2564         0.0663         0.2581     True
AXON         0.3245       0.0690       0.3344       0.0281         0.3347        -0.0223    False
BA           0.0115      -0.0746      -0.1450      -0.1610         0.3450         0.0479     True
CARR         0.1414       0.1540       0.1865       0.1177        -0.0329         0.0999     True


Quick check on one firm by hand:

In [5]:
r = mx.loc["CAT"]
g23 = r["revenue_fy_minus_2"] / r["revenue_fy_minus_3"] - 1
g24 = r["revenue_fy_minus_1"] / r["revenue_fy_minus_2"] - 1
m3 = [r["margin_fy_minus_3"], r["margin_fy_minus_2"], r["margin_fy_minus_1"]]

print(f"CAT revenue  {r['revenue_fy_minus_3']/1e9:.3f}  {r['revenue_fy_minus_2']/1e9:.3f}"
      f"  {r['revenue_fy_minus_1']/1e9:.3f}  ($bn, FY22-24)")
print(f"growth: ({g23:.4f} + {g24:.4f}) / 2 = {(g23 + g24) / 2:.4f}"
      f"   table {fc.loc['CAT', 'base_growth']:.4f}")
print(f"margin: mean of {[round(float(x), 4) for x in m3]} = {np.mean(m3):.4f}"
      f"   table {fc.loc['CAT', 'base_margin']:.4f}")

CAT revenue  59.427  67.060  64.809  ($bn, FY22-24)
growth: (0.1284 + -0.0336) / 2 = 0.0474   table 0.0474
margin: mean of [0.133, 0.1933, 0.2017] = 0.1760   table 0.1760


### How accurate is the baseline?

Mean absolute error vs. actual FY2025. Person 4 does the full scoring, this is just the two
benchmarks.

In [6]:
GROUPS = {
    "all firms":  pd.Series(True, index=fc.index),
    "backlog":    fc["backlog"],
    "no backlog": ~fc["backlog"],
    "ex-flagged": fc["flag"] == "",
}


def mae(pred, actual):
    return (pred - actual).abs().mean()


acc = pd.DataFrame({
    name: {
        "n": int(mask.sum()),
        "growth MAE, 3-yr":  mae(fc.loc[mask, "base_growth"], fc.loc[mask, "actual_growth"]),
        "growth MAE, last":  mae(fc.loc[mask, "last_growth"], fc.loc[mask, "actual_growth"]),
        "growth bias, 3-yr": (fc.loc[mask, "base_growth"] - fc.loc[mask, "actual_growth"]).mean(),
        "margin MAE, 3-yr":  mae(fc.loc[mask, "base_margin"], fc.loc[mask, "actual_margin"]),
        "margin MAE, last":  mae(fc.loc[mask, "last_margin"], fc.loc[mask, "actual_margin"]),
    }
    for name, mask in GROUPS.items()
})
print(acc.to_string(float_format=lambda v: f"{v:.4f}"))

                   all firms  backlog  no backlog  ex-flagged
n                    53.0000  38.0000     15.0000     49.0000
growth MAE, 3-yr      0.0651   0.0747      0.0410      0.0542
growth MAE, last      0.0561   0.0659      0.0314      0.0498
growth bias, 3-yr     0.0108   0.0182     -0.0077      0.0280
margin MAE, 3-yr      0.0282   0.0242      0.0383      0.0236
margin MAE, last      0.0187   0.0195      0.0167      0.0154


The three-year average misses backlog firms by almost twice as much as the others (7.5% vs
4.1%), and it tends to overshoot them. That gap is what the model is supposed to close, and
the 15 firms without backlog work as a control group.

Last year beats the three-year average on both growth and margin. The rubric asks for the
three-year average so that stays the official baseline, but we report both.

A few of the biggest misses come from acquisitions or divestitures rather than backlog, like
XYL (Evoqua in FY23) and CARR. If the model gets those right from the MD&A it's right for a
different reason, so Person 4 should keep them separate.

In [7]:
fc["base_miss"] = fc["base_growth"] - fc["actual_growth"]
worst = fc.reindex(fc["base_miss"].abs().sort_values(ascending=False).index).head(8)
print(worst[["base_growth", "actual_growth", "base_miss", "backlog"]]
      .round(3).to_string())

fc.drop(columns="base_miss").to_csv(os.path.join(OUT, "baseline_forecasts.csv"))
print(f"\nsaved {OUT}/baseline_forecasts.csv  ({len(fc)} firms)")

        base_growth  actual_growth  base_miss  backlog
ticker                                                
BA            0.011          0.345     -0.333     True
DD           -0.238          0.019     -0.257     True
PCAR          0.089         -0.155      0.244     True
XYL           0.248          0.055      0.193     True
TEX          -0.049          0.131     -0.180     True
CARR          0.141         -0.033      0.174     True
OSK           0.139         -0.029      0.167     True
FTV          -0.143          0.019     -0.162    False

saved output/baseline_forecasts.csv  (53 firms)


## Part 2: DCF, tested on dummy numbers

### npv from Lecture 6

In [8]:
def npv(rate, cashflows, first_at=1):
    """first_at=1: the first flow arrives one year from now."""
    total = 0.0
    for t, c in enumerate(cashflows):
        years = t + first_at
        total = total + c / (1 + rate) ** years
    return total


mine = npv(0.10, [100, 100, 100])
try:
    import numpy_financial as npf
    theirs = npf.npv(0.10, [0, 100, 100, 100])   # npf puts flows[0] at t = 0
    print(f"mine    : {mine:,.4f}")
    print(f"npf.npv : {theirs:,.4f}")
    print(f"agree   : {abs(mine - theirs) < 1e-9}")
except ImportError:
    print(f"mine: {mine:,.4f}   (numpy_financial not installed; expect 248.6852)")

mine    : 248.6852
npf.npv : 248.6852
agree   : True


### Fade and reinvestment

Both the baseline and the model only forecast FY2025, so the DCF uses the forecast for year 1
and then fades growth in a straight line to `G_TERM` by year 5. If we held AXON's 32.5%
baseline for five years it would end up about four times its current size, and that
compounding would swamp every comparison.

Reinvestment is `growth / ROIC`, the share of after-tax operating profit that has to go back
into the business to grow at that rate. With a fixed reinvestment rate, a higher growth
forecast would get its extra value for free. It's floored at zero so a shrinking firm doesn't
get extra cash out of shrinking.

In [9]:
def growth_path(g1, g_term=G_TERM, years=YEARS, fade=True):
    """Year 1 = the forecast, then a straight line down to g_term by year 5.
    fade=False keeps growth flat all 5 years (only used for the comparison in Part 3).
    """
    path = []
    for t in range(1, years + 1):
        w = (t - 1) / (years - 1) if fade else 0.0   # weight on g_term, 0 in year 1
        path.append((1 - w) * g1 + w * g_term)
    return path


print("fade from 12%:", "  ".join(f"{g:.2%}" for g in growth_path(0.12)))
print("fade from -8%:", "  ".join(f"{g:.2%}" for g in growth_path(-0.08)))

fade from 12%: 12.00%  9.62%  7.25%  4.88%  2.50%
fade from -8%: -8.00%  -5.38%  -2.75%  -0.12%  2.50%


In [10]:
def dcf(rev0, growth, margin, wacc=WACC, g_term=G_TERM, tax=TAX,
        roic=ROIC, years=YEARS, fade=True, detail=False):
    """EV from FY2024 revenue, a year-1 growth rate and a margin."""
    if g_term >= wacc:
        raise ValueError(f"g_term {g_term:.2%} must be below WACC {wacc:.2%}")

    # 1. project revenue (forecast in year 1, then fade to g_term)
    path = growth_path(growth, g_term, years, fade)
    rev, revs = rev0, []
    for g in path:
        rev = rev * (1 + g)
        revs.append(rev)

    # 2. FCF = NOPAT minus reinvestment
    fcf = []
    for g, r in zip(path, revs):
        nopat = r * margin * (1 - tax)
        reinvest = max(g, 0) / roic        # floored at 0 for down years
        fcf.append(nopat * (1 - reinvest))

    # 3. discount back to early 2025
    pv_e = npv(wacc, fcf)

    # 4. terminal value after year 5, discounted back too
    fcf_next = revs[-1] * (1 + g_term) * margin * (1 - tax) * (1 - g_term / roic)
    tv = fcf_next / (wacc - g_term)
    pv_t = tv / (1 + wacc) ** years

    ev = pv_e + pv_t
    if detail:
        return {"growth": path, "revenue": revs, "fcf": fcf,
                "pv_explicit": pv_e, "pv_terminal": pv_t, "ev": ev}
    return ev

### Test 1: dummy firm

Revenue of 1,000, 5% growth in year 1, 15% margin. The table is rebuilt with numpy instead of
the loop, and the two answers should match.

In [11]:
REV, G1, M = 1000.0, 0.05, 0.15
d = dcf(REV, G1, M, detail=True)

t = np.arange(1, YEARS + 1)
g = np.linspace(G1, G_TERM, YEARS)                 # same fade, done with numpy
rev = REV * np.cumprod(1 + g)
nopat = rev * M * (1 - TAX)
fcf = nopat * (1 - np.maximum(g, 0) / ROIC)
disc = 1 / (1 + WACC) ** t
tv = rev[-1] * (1 + G_TERM) * M * (1 - TAX) * (1 - G_TERM / ROIC) / (WACC - G_TERM)
ev_check = (fcf * disc).sum() + tv * disc[-1]

table = pd.DataFrame({"growth": g, "revenue": rev, "NOPAT": nopat,
                      "reinvest %": np.maximum(g, 0) / ROIC, "FCF": fcf,
                      "discount": disc, "PV": fcf * disc},
                     index=pd.Index([f"FY{2024 + k}" for k in t], name="year"))
print(table.round(4).to_string())
print(f"\nPV of 5 years    : {d['pv_explicit']:>9,.2f}")
print(f"PV of terminal   : {d['pv_terminal']:>9,.2f}   ({d['pv_terminal'] / d['ev']:.1%} of EV)")
print(f"enterprise value : {d['ev']:>9,.2f}")
print(f"vectorised check : {ev_check:>9,.2f}   agree {abs(ev_check - d['ev']) < 1e-9}")

        growth    revenue     NOPAT  reinvest %       FCF  discount       PV
year                                                                        
FY2025  0.0500  1050.0000  118.1250      0.2500   88.5938    0.9191  81.4281
FY2026  0.0438  1095.9375  123.2930      0.2188   96.3226    0.8448  81.3712
FY2027  0.0375  1137.0352  127.9165      0.1875  103.9321    0.7764  80.6981
FY2028  0.0312  1172.5675  131.9138      0.1562  111.3023    0.7136  79.4307
FY2029  0.0250  1201.8817  135.2117      0.1250  118.3102    0.6559  77.6029

PV of 5 years    :    400.53
PV of terminal   :  1,262.59   (75.9% of EV)
enterprise value :  1,663.12
vectorised check :  1,663.12   agree True


### Test 2: Gordon identity

If year-1 growth is already `G_TERM`, every year grows at `G_TERM` and the DCF should equal
`FCF₁ / (WACC − g)`.

In [12]:
fcf1 = REV * (1 + G_TERM) * M * (1 - TAX) * (1 - G_TERM / ROIC)
gordon = fcf1 / (WACC - G_TERM)
model = dcf(REV, G_TERM, M)

print(f"Gordon, one line : {gordon:,.6f}")
print(f"dcf()            : {model:,.6f}")
print(f"difference       : {model - gordon:.1e}")

Gordon, one line : 1,601.562500
dcf()            : 1,601.562500
difference       : -9.1e-13


### Test 3: guards and scaling

In [13]:
# g_term >= WACC should throw an error
try:
    dcf(REV, G1, M, g_term=0.09)
except ValueError as e:
    print("ValueError:", e)

# EV should scale 1:1 with margin and with revenue
print(f"\ndouble the margin  -> x{dcf(REV, G1, 2 * M) / dcf(REV, G1, M):.6f}")
print(f"double the revenue -> x{dcf(2 * REV, G1, M) / dcf(REV, G1, M):.6f}")

# down year: reinvestment is 0, so FCF should equal NOPAT
s = dcf(REV, -0.10, M, detail=True)
yr1_nopat = s["revenue"][0] * M * (1 - TAX)
print(f"\n-10% year: FCF {s['fcf'][0]:,.2f}  NOPAT {yr1_nopat:,.2f}"
      f"  equal {abs(s['fcf'][0] - yr1_nopat) < 1e-9}")

# ROIC > WACC, so more growth should always mean more value
vals = [dcf(REV, x, M) for x in [-0.05, 0.0, 0.05, 0.10, 0.15]]
print(f"value rises with growth: {all(np.diff(vals) > 0)}")

ValueError: g_term 9.00% must be below WACC 8.80%

double the margin  -> x2.000000
double the revenue -> x2.000000

-10% year: FCF 101.25  NOPAT 101.25  equal True
value rises with growth: True


Since every cash flow gets multiplied by the margin, EV is proportional to margin. So any
valuation error splits exactly into a growth part and a margin part:

`(1 + total error) = (1 + growth part) × (1 + margin part)`

The margin part is just `forecast margin / actual margin − 1`. We use this in Part 3 to see
which assumption drove the gap.

## Part 3: All 53 firms

Each firm is valued at early 2025 from its FY2024 revenue, once per forecast. The yardstick is
the same DCF fed what actually happened in FY2025 (actual growth and actual margin), saved as
`ev_perfect_foresight`. It is not a market value: every assumption other than the forecast is
identical on both sides, so the gap between a forecast's value and the yardstick comes only from
the forecast. Every other forecast is scored by how far off its value is.

We run two versions. In the growth-only version every scenario uses the same margin, so the
only difference is the growth forecast. That's the cleanest test of the backlog idea. (The %
error comes out the same whichever margin you hold fixed, since EV is linear in margin.) In
the full version each scenario uses its own growth and margin.

In [14]:
def value_all(growth, margin, **kw):
    """EV for every firm from one set of year-1 forecasts."""
    out = {}
    for tk in fc.index:
        out[tk] = dcf(mx.loc[tk, "revenue_fy_minus_1"], growth[tk], margin[tk], **kw)
    return pd.Series(out).rename_axis("ticker")


def score(growth, margin, **kw):
    """Valuation errors for one forecast vs. the actual-results value."""
    true_full = value_all(fc["actual_growth"], fc["actual_margin"], **kw)
    hold = fc["base_margin"]                               # any fixed margin gives the same %
    g_only = value_all(growth, hold, **kw) / value_all(fc["actual_growth"], hold, **kw) - 1
    full = value_all(growth, margin, **kw) / true_full - 1
    m_part = margin / fc["actual_margin"] - 1
    return pd.DataFrame({"ev_true": true_full, "growth_only": g_only,
                         "full": full, "margin_part": m_part})


SCEN = {
    "3-yr average": (fc["base_growth"], fc["base_margin"]),
    "last year":    (fc["last_growth"], fc["last_margin"]),
}

### Adding Person 3's forecasts

Save a file at `data/lm_forecasts.csv` with one row per firm per run:

| column | meaning |
| :--- | :--- |
| `ticker` | same as `master.csv` |
| `label` | e.g. `AI full`, `AI no MD&A` |
| `growth` | forecast FY2025 revenue growth as a decimal (0.08 = 8%) |
| `margin` | forecast FY2025 operating margin as a decimal |

Then rerun the notebook. Every table after this picks up the new runs.

In [15]:
LM_FILE = os.path.join("..", "data", "lm_forecasts.csv")

if os.path.exists(LM_FILE):
    lm = pd.read_csv(LM_FILE)
    for label, part in lm.groupby("label"):
        part = part.set_index("ticker").reindex(fc.index)
        missing = part["growth"].isna().sum()
        if missing:
            print(f"{label}: {missing} firms missing -- they take the baseline")
        g = part["growth"].fillna(fc["base_growth"])
        m = part["margin"].fillna(fc["base_margin"])
        SCEN[label] = (g, m)
    print("loaded:", [k for k in SCEN])
else:
    print(f"no {LM_FILE} yet -- scoring the two benchmarks only")

loaded: ['3-yr average', 'last year', 'AI full', 'AI no MD&A']


### Forecast accuracy, every scenario

Mean absolute error against actual FY2025, by group. Person 4 owns the full scoring; this is
here so the valuation results below can be read next to the forecast errors that drive them.

In [16]:
def acc_table(scen, mask):
    rows = {}
    for name, (g, m) in scen.items():
        rows[name] = {
            "n": int(mask.sum()),
            "growth MAE":   (g - fc["actual_growth"]).abs()[mask].mean(),
            "growth bias":  (g - fc["actual_growth"])[mask].mean(),
            "margin MAE":   (m - fc["actual_margin"]).abs()[mask].mean(),
        }
    return pd.DataFrame(rows)


for gname, mask in GROUPS.items():
    print(f"--- {gname} " + "-" * (60 - len(gname)))
    print(acc_table(SCEN, mask).to_string(float_format=lambda v: f"{v:.4f}"))
    print()

--- all firms ---------------------------------------------------
             3-yr average  last year  AI full  AI no MD&A
n                 53.0000    53.0000  53.0000     53.0000
growth MAE         0.0651     0.0561   0.0477      0.0527
growth bias        0.0108     0.0001  -0.0107     -0.0066
margin MAE         0.0282     0.0187   0.0179      0.0178

--- backlog -----------------------------------------------------
             3-yr average  last year  AI full  AI no MD&A
n                 38.0000    38.0000  38.0000     38.0000
growth MAE         0.0747     0.0659   0.0517      0.0579
growth bias        0.0182    -0.0054  -0.0134     -0.0094
margin MAE         0.0242     0.0195   0.0170      0.0178

--- no backlog --------------------------------------------------
             3-yr average  last year  AI full  AI no MD&A
n                 15.0000    15.0000  15.0000     15.0000
growth MAE         0.0410     0.0314   0.0377      0.0395
growth bias       -0.0077     0.0142  -0.0038 

With 53 firms (and 15 without backlog), a gap of a point or two can be luck. A paired bootstrap
resamples firms with replacement and asks how often the gap survives: if the 95% range for
"A's error minus B's error" stays below zero, A is reliably more accurate.

In [17]:
def paired_boot(err_a, err_b, draws=10000, seed=0):
    """Mean of (err_a - err_b) and its 95% bootstrap range, resampling firms."""
    d = (err_a - err_b).to_numpy()
    rng = np.random.default_rng(seed)
    means = []
    for _ in range(draws):
        means.append(rng.choice(d, len(d)).mean())
    return d.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)


def gerr(name):
    return (SCEN[name][0] - fc["actual_growth"]).abs()


pairs = [(a, b) for a in SCEN for b in SCEN if a.startswith("AI full") and b != a]
for gname in ["all firms", "backlog", "no backlog"]:
    mask = GROUPS[gname]
    for a, b in pairs:
        mean, lo, hi = paired_boot(gerr(a)[mask], gerr(b)[mask])
        verdict = "reliably better" if hi < 0 else ("reliably worse" if lo > 0 else "not distinguishable")
        print(f"{gname:<11} {a} vs {b:<12} growth error {mean * 100:+.2f}pp"
              f"  95% [{lo * 100:+.2f}, {hi * 100:+.2f}]  {verdict}")
    print()

all firms   AI full vs 3-yr average growth error -1.74pp  95% [-3.57, -0.02]  reliably better


all firms   AI full vs last year    growth error -0.84pp  95% [-2.23, +0.44]  not distinguishable


all firms   AI full vs AI no MD&A   growth error -0.50pp  95% [-1.24, +0.18]  not distinguishable



backlog     AI full vs 3-yr average growth error -2.30pp  95% [-4.53, -0.23]  reliably better
backlog     AI full vs last year    growth error -1.42pp  95% [-3.20, +0.19]  not distinguishable


backlog     AI full vs AI no MD&A   growth error -0.62pp  95% [-1.53, +0.20]  not distinguishable



no backlog  AI full vs 3-yr average growth error -0.33pp  95% [-3.42, +2.44]  not distinguishable


no backlog  AI full vs last year    growth error +0.63pp  95% [-0.98, +2.35]  not distinguishable
no backlog  AI full vs AI no MD&A   growth error -0.18pp  95% [-1.29, +0.95]  not distinguishable



What the clean run says (all 53 firms, with the 9 non-December filers on their fiscal-2024 10-Ks):

The full-filing model cuts growth error from 6.5% to 4.8% against the three-year average, and the
bootstrap range stays below zero both overall and for backlog firms. Against "last year" (5.6%) the
gap is not distinguishable from luck. On the 15 firms without backlog the model has no edge at all,
which is the pattern the backlog thesis predicts. Removing the MD&A raises growth error to 5.3%, but
that gap is not distinguishable either.

On margins, the model and "last year" are tied (about 1.8%) and both well ahead of the three-year
average (2.8%). The model's forecasts mostly track the latest margin.

In [18]:
res = {name: score(g, m) for name, (g, m) in SCEN.items()}

ev_true = res["3-yr average"]["ev_true"]
bad = ev_true[ev_true <= 0]
print(f"firms whose actual-results value is not positive: {len(bad)}  {list(bad.index)}")
print("  (can't take a % error against a negative value, so these are left out of 'full')")

firms whose actual-results value is not positive: 1  ['AXON']
  (can't take a % error against a negative value, so these are left out of 'full')


In [19]:
def summarise(res, mask):
    rows = {}
    for name, r in res.items():
        ok = mask & (r["ev_true"] > 0)
        rows[name] = {
            "n": int(mask.sum()),
            "growth only, mean |err|":   r.loc[mask, "growth_only"].abs().mean(),
            "growth only, median |err|": r.loc[mask, "growth_only"].abs().median(),
            "full, n": int(ok.sum()),
            "full, mean |err|":   r.loc[ok, "full"].abs().mean(),
            "full, median |err|": r.loc[ok, "full"].abs().median(),
        }
    return pd.DataFrame(rows)


for gname, mask in GROUPS.items():
    print(f"--- {gname} " + "-" * (60 - len(gname)))
    print(summarise(res, mask).to_string(float_format=lambda v: f"{v:.3f}"))
    print()

--- all firms ---------------------------------------------------
                           3-yr average  last year  AI full  AI no MD&A
n                                53.000     53.000   53.000      53.000
growth only, mean |err|           0.111      0.090    0.076       0.084
growth only, median |err|         0.053      0.044    0.052       0.055
full, n                          52.000     52.000   52.000      52.000
full, mean |err|                  0.256      0.200    0.212       0.217
full, median |err|                0.114      0.091    0.095       0.087

--- backlog -----------------------------------------------------
                           3-yr average  last year  AI full  AI no MD&A
n                                38.000     38.000   38.000      38.000
growth only, mean |err|           0.127      0.106    0.083       0.092
growth only, median |err|         0.059      0.076    0.052       0.062
full, n                          38.000     38.000   38.000      38.000
ful

Reading the table: a growth-only median of 0.053 means the three-year average's growth
forecast puts the median firm's value 5.3% away from its actual-results value. Means are a lot
higher than medians because a few firms (BA, PCAR, CARR, MMM) are off by 30% or more, so we
lead with medians.

The two benchmarks don't rank the same way in every group. Last year wins overall, but the
three-year average has the lower median for backlog firms and for the ex-flagged set, so the
write-up needs to say which group it's talking about.

### Which assumption drove the DCF gap?

This splits the full error into its growth part and margin part for each forecast. We use
medians because a few firms with tiny or negative margins (BA, AXON, DD) give percentage
errors in the hundreds.

In [20]:
split = {}
for name, r in res.items():
    ok = (r["ev_true"] > 0) & (fc["flag"] == "")
    split[name] = {
        "n": int(ok.sum()),
        "median |total|":       r.loc[ok, "full"].abs().median(),
        "median |growth part|": r.loc[ok, "growth_only"].abs().median(),
        "median |margin part|": r.loc[ok, "margin_part"].abs().median(),
        "growth part bigger":   (r.loc[ok, "growth_only"].abs()
                                 > r.loc[ok, "margin_part"].abs()).mean(),
    }
print("ex-flagged, positive actual value\n")
print(pd.DataFrame(split).to_string(float_format=lambda v: f"{v:.3f}"))

# check the split holds for every firm
r = res["3-yr average"]
ok = r["ev_true"] > 0
lhs = 1 + r.loc[ok, "full"]
rhs = (1 + r.loc[ok, "growth_only"]) * (1 + r.loc[ok, "margin_part"])
print(f"\n(1+total) = (1+growth)(1+margin) on every firm: {np.allclose(lhs, rhs)}")

ex-flagged, positive actual value

                      3-yr average  last year  AI full  AI no MD&A
n                           48.000     48.000   48.000      48.000
median |total|               0.108      0.091    0.095       0.087
median |growth part|         0.049      0.060    0.052       0.057
median |margin part|         0.115      0.063    0.061       0.062
growth part bigger           0.292      0.438    0.417       0.417

(1+total) = (1+growth)(1+margin) on every firm: True


For the three-year average the margin miss matters more than the growth miss. It's the bigger
of the two for about 70% of firms.

Part of that is just math. EV is proportional to margin, so missing a 15% margin by 1.5 points
is already a 10% value error. The other part comes from our setup: growth only drives year 1
before fading, and the terminal value doesn't depend on it at all.

Since that second part is a modelling choice, here's the same split without the fade:

In [21]:
fade_cmp = {}
for label, kw in [("fade (base)", {}), ("no fade", {"fade": False})]:
    r = score(fc["base_growth"], fc["base_margin"], **kw)
    ok = (r["ev_true"] > 0) & (fc["flag"] == "")
    fade_cmp[label] = {
        "median |growth part|": r.loc[ok, "growth_only"].abs().median(),
        "median |margin part|": r.loc[ok, "margin_part"].abs().median(),
        "growth part bigger":   (r.loc[ok, "growth_only"].abs()
                                 > r.loc[ok, "margin_part"].abs()).mean(),
    }
print("3-yr average, ex-flagged\n")
print(pd.DataFrame(fade_cmp).to_string(float_format=lambda v: f"{v:.3f}"))

3-yr average, ex-flagged

                      fade (base)  no fade
median |growth part|        0.049    0.096
median |margin part|        0.115    0.115
growth part bigger          0.292    0.500


Without the fade the growth part about doubles, which makes sense. But holding a one-year
forecast for five years assumes something neither the baseline nor the model is actually
claiming, so we kept the fade and will note in the memo that the split depends on it.

In [22]:
r = res["3-yr average"].join(fc[["base_growth", "actual_growth",
                                 "base_margin", "actual_margin"]])
r = r[r["ev_true"] > 0]
show = r.reindex(r["full"].abs().sort_values(ascending=False).index).head(8)
print("largest 3-yr-average valuation errors\n")
print(show[["base_growth", "actual_growth", "base_margin", "actual_margin",
            "growth_only", "margin_part", "full"]].round(3).to_string())

largest 3-yr-average valuation errors

        base_growth  actual_growth  base_margin  actual_margin  growth_only  margin_part   full
ticker                                                                                         
BA            0.011          0.345       -0.075          0.048       -0.370       -2.559 -1.982
PCAR          0.089         -0.155        0.152          0.106        0.656        0.432  1.372
MMM          -0.030          0.015       -0.024          0.186       -0.083       -1.128 -1.117
CARR          0.141         -0.033        0.154          0.100        0.321        0.542  1.037
DE            0.005         -0.117        0.189          0.137        0.323        0.380  0.826
GNRC         -0.025         -0.020        0.115          0.069       -0.011        0.674  0.655
DD           -0.238          0.019        0.029          0.029       -0.465       -0.013 -0.472
VRT           0.187          0.277        0.112          0.179       -0.114       -0.372 -0.443


In [23]:
long = []
for name, r in res.items():
    g, m = SCEN[name]
    part = pd.DataFrame({"scenario": name, "growth": g, "margin": m,
                         "ev": value_all(g, m), "ev_perfect_foresight": r["ev_true"],
                         "err_growth_only": r["growth_only"],
                         "err_margin_part": r["margin_part"],
                         "err_full": r["full"]})
    long.append(part)
long = pd.concat(long).rename_axis("ticker").reset_index()
long.to_csv(os.path.join(OUT, "dcf_valuations.csv"), index=False)
print(f"saved {OUT}/dcf_valuations.csv  ({len(long)} rows, {long['scenario'].nunique()} scenarios)")

saved output/dcf_valuations.csv  (212 rows, 4 scenarios)


### A mechanical backlog rule, on 28 firms

To check whether the model adds anything beyond finding the backlog number, a fixed rule was applied
to 28 firms (19 with backlog, 9 without): use company-wide FY2025 revenue guidance if the 10-K gives
it; otherwise average FY2024 growth with the year-over-year change in backlog (or orders); otherwise
FY2024 growth; round to 0.5%. Margin is the FY2024 margin, adjusted only for a quantified FY2024
impairment or legal charge. The rule was applied with Claude and checked by the team; the sentence
behind every number is in `data/rule_labels_evidence.csv`.

Every scenario is scored here on the same 28 firms, so the rows compare like with like. These firms
are not a random draw (the most dramatic movers were left out), so compare rows within this table,
not against the 53-firm tables above.

In [24]:
RULE_FILE = os.path.join("..", "data", "rule_labels.csv")

if os.path.exists(RULE_FILE):
    rule = pd.read_csv(RULE_FILE).set_index("ticker")
    sub = rule.index
    rg = rule["growth"].reindex(fc.index).fillna(fc["base_growth"])
    rm = rule["margin"].reindex(fc.index).fillna(fc["base_margin"])
    scen28 = dict(SCEN)
    scen28["backlog rule"] = (rg, rm)
    in28 = pd.Series(fc.index.isin(sub), index=fc.index)

    rows = {}
    for name, (g, m) in scen28.items():
        r = score(g, m)
        ok = in28 & (r["ev_true"] > 0)
        rows[name] = {
            "growth MAE, 28":        (g - fc["actual_growth"]).abs()[in28].mean(),
            "growth MAE, backlog":   (g - fc["actual_growth"]).abs()[in28 & fc["backlog"]].mean(),
            "growth MAE, no backlog": (g - fc["actual_growth"]).abs()[in28 & ~fc["backlog"]].mean(),
            "margin MAE":            (m - fc["actual_margin"]).abs()[in28].mean(),
            "val err, growth only":  r.loc[in28, "growth_only"].abs().median(),
            "val err, full":         r.loc[ok, "full"].abs().median(),
        }
    print(f"{in28.sum()} firms, {(in28 & fc['backlog']).sum()} with backlog\n")
    print(pd.DataFrame(rows).to_string(float_format=lambda v: f"{v:.4f}"))

    rerr = (rg - fc["actual_growth"]).abs()
    print()
    for gname, mask in [("28 firms", in28), ("backlog", in28 & fc["backlog"])]:
        mean, lo, hi = paired_boot(gerr("AI full")[mask], rerr[mask])
        print(f"{gname:<9} AI full vs backlog rule: {mean * 100:+.2f}pp  95% [{lo * 100:+.2f}, {hi * 100:+.2f}]")
else:
    print(f"no {RULE_FILE} -- skipping the rule comparison")

28 firms, 19 with backlog

                        3-yr average  last year  AI full  AI no MD&A  backlog rule
growth MAE, 28                0.0338     0.0327   0.0305      0.0272        0.0388
growth MAE, backlog           0.0352     0.0359   0.0281      0.0258        0.0464
growth MAE, no backlog        0.0307     0.0261   0.0358      0.0303        0.0227
margin MAE                    0.0170     0.0102   0.0098      0.0095        0.0097
val err, growth only          0.0431     0.0357   0.0358      0.0277        0.0431
val err, full                 0.1049     0.0739   0.0832      0.0725        0.0878



28 firms  AI full vs backlog rule: -0.83pp  95% [-2.12, +0.30]


backlog   AI full vs backlog rule: -1.84pp  95% [-3.48, -0.41]


In [25]:
if os.path.exists(RULE_FILE):
    step = rule["step"]
    by_step = pd.DataFrame({
        "firms": step.value_counts(),
        "rule growth MAE": rerr[sub].groupby(step).mean(),
        "AI full growth MAE": gerr("AI full")[sub].groupby(step).mean(),
    })
    print(by_step.round(4).to_string())
    show = pd.DataFrame({"rule": rg[sub], "AI full": SCEN["AI full"][0][sub],
                         "actual": fc["actual_growth"][sub], "step": step})
    show["rule miss"] = (show["rule"] - show["actual"]).abs()
    print("\nlargest rule misses\n")
    print(show.sort_values("rule miss", ascending=False).head(5).round(3).to_string())

      firms  rule growth MAE  AI full growth MAE
step                                            
G1        1           0.0269              0.0169
G2       12           0.0458              0.0227
G3       15           0.0340              0.0377

largest rule misses

         rule  AI full  actual step  rule miss
ticker                                        
PNR    -0.090     0.02   0.023   G2      0.113
HUBB   -0.070     0.02   0.038   G2      0.108
LECO   -0.045     0.02   0.056   G3      0.101
LII     0.070     0.08  -0.027   G3      0.097
WSO     0.045     0.05  -0.050   G3      0.095


## Part 4: Sensitivity

### Growth vs. margin

EV per dollar of FY2024 revenue, with everything else at base. The axes cover roughly the
middle of the sample.

In [26]:
growths = [-0.05, 0.00, 0.05, 0.10, 0.15]
margins = [0.10, 0.125, 0.15, 0.175, 0.20]

grid = pd.DataFrame(
    [[dcf(1.0, g, m) for g in growths] for m in margins],
    index=[f"{m:.1%}" for m in margins],
    columns=[f"{g:.0%}" for g in growths])
grid.index.name = "margin"
grid.columns.name = "year-1 growth"
print("EV / FY2024 revenue\n")
print(grid.round(2).to_string())

print(f"\nmedian firm, 3-yr inputs: growth {fc['base_growth'].median():.1%},"
      f" margin {fc['base_margin'].median():.1%}"
      f" -> {dcf(1.0, fc['base_growth'].median(), fc['base_margin'].median()):.2f}x revenue")

EV / FY2024 revenue

year-1 growth   -5%    0%    5%   10%   15%
margin                                     
10.0%          0.92  1.03  1.11  1.19  1.28
12.5%          1.15  1.28  1.39  1.49  1.60
15.0%          1.38  1.54  1.66  1.79  1.92
17.5%          1.61  1.80  1.94  2.09  2.24
20.0%          1.84  2.06  2.22  2.39  2.56

median firm, 3-yr inputs: growth 6.7%, margin 15.2% -> 1.73x revenue


For reference, Damodaran's January 2026 EV/Sales for Machinery is 3.43x, about double what we
get for the median firm, so the market is pricing in more growth or a lower discount rate than
we assume. It doesn't affect our comparison because every scenario is scored as a ratio to the
actual-results value and the level drops out. These shouldn't be read as price targets though.

### One assumption at a time

Ranking inputs by swing alone mostly ranks the ranges you picked (Lecture 8), so the table also
shows each range and the swing per percentage point. Base case is the median firm's three-year
inputs.

In [27]:
G0, M0 = fc["base_growth"].median(), fc["base_margin"].median()
base_v = dcf(1.0, G0, M0)

TESTS = [   # (label, input, low, high)
    ("year-1 growth",   "growth", G0 - 0.05, G0 + 0.05),
    ("margin",          "margin", M0 - 0.03, M0 + 0.03),
    ("WACC",            "wacc",   WACC - 0.01, WACC + 0.01),
    ("terminal growth", "g_term", G_TERM - 0.01, G_TERM + 0.01),
    ("tax rate",        "tax",    0.20, 0.30),
    ("ROIC",            "roic",   0.15, 0.25),
]


def value_with(kw, x):
    args = {"growth": G0, "margin": M0}
    if kw in args:
        args[kw] = x
        return dcf(1.0, args["growth"], args["margin"])
    return dcf(1.0, G0, M0, **{kw: x})


rows = []
for name, kw, lo, hi in TESTS:
    v_lo, v_hi = value_with(kw, lo), value_with(kw, hi)
    width = (hi - lo) * 100
    rows.append({"input": name, "low": lo, "high": hi, "range (pp)": width,
                 "value low": v_lo, "value high": v_hi,
                 "swing": abs(v_hi - v_lo), "swing % of base": abs(v_hi - v_lo) / base_v,
                 "swing per pp": abs(v_hi - v_lo) / width})
tor = pd.DataFrame(rows).set_index("input").sort_values("swing", ascending=False)

print(f"base value {base_v:.3f}x revenue  (growth {G0:.1%}, margin {M0:.1%})\n")
print(tor.to_string(float_format=lambda v: f"{v:.3f}"))
print("\nranked by swing      :", list(tor.index))
print("ranked by swing / pp :", list(tor.sort_values("swing per pp", ascending=False).index))

base value 1.731x revenue  (growth 6.7%, margin 15.2%)

                  low  high  range (pp)  value low  value high  swing  swing % of base  swing per pp
input                                                                                               
margin          0.122 0.182       6.000      1.390       2.072  0.682            0.394         0.114
WACC            0.078 0.098       2.000      2.069       1.486  0.583            0.337         0.291
terminal growth 0.015 0.035       2.000      1.581       1.934  0.353            0.204         0.177
year-1 growth   0.017 0.117      10.000      1.605       1.861  0.256            0.148         0.026
tax rate        0.200 0.300      10.000      1.846       1.615  0.231            0.133         0.023
ROIC            0.150 0.250      10.000      1.627       1.793  0.166            0.096         0.017

ranked by swing      : ['margin', 'WACC', 'terminal growth', 'year-1 growth', 'tax rate', 'ROIC']
ranked by swing / pp : ['WACC', 'term

By raw swing, margin comes out on top, but it was moved 6 points and WACC only 2. Per point,
WACC and terminal growth move value the most, then margin, and year-1 growth is a distant
fourth. One point of year-1 growth is worth about a quarter of a point of margin because it
only drives one year before fading.

WACC and terminal growth are the same for every forecast, so they can't decide which forecast
wins. The next table checks that.

### Can the fixed assumptions change who wins?

What matters for the project is which forecast lands closer to the actual-results value, so
this reruns the 53-firm scoring under each alternative setting.

In [28]:
ke_dam = RF + BETA * 0.0433                    # Damodaran implied ERP, Jan 2025
wacc_dam = round((1 - D_WT) * ke_dam + D_WT * KD_AT, 3)

SETTINGS = {
    "base":                     {},
    "WACC -1pp":                {"wacc": WACC - 0.01},
    "WACC +1pp":                {"wacc": WACC + 0.01},
    f"WACC {wacc_dam:.1%} (ERP 4.33%)": {"wacc": wacc_dam},
    "terminal growth 1.5%":     {"g_term": 0.015},
    "terminal growth 3.5%":     {"g_term": 0.035},
    "tax 21%":                  {"tax": 0.21},
    "ROIC 15%":                 {"roic": 0.15},
    "ROIC 25%":                 {"roic": 0.25},
    "no fade (hold 5 years)":   {"fade": False},
}

rows = {}
for sname, kw in SETTINGS.items():
    row = {}
    for name, (g, m) in SCEN.items():
        r = score(g, m, **kw)
        row[name] = r["growth_only"].abs().median()
    row["winner"] = min(SCEN, key=lambda k: row[k])
    rows[sname] = row
rob = pd.DataFrame(rows).T

print("median |valuation error|, growth only, all 53 firms\n")
print(rob.to_string(float_format=lambda v: f"{v:.4f}"))
print(f"\nwinner the same in every setting: {rob['winner'].nunique() == 1}")

median |valuation error|, growth only, all 53 firms

                       3-yr average last year AI full AI no MD&A     winner
base                         0.0533    0.0440  0.0518     0.0553  last year
WACC -1pp                    0.0580    0.0473  0.0554     0.0600  last year
WACC +1pp                    0.0486    0.0408  0.0473     0.0507  last year
WACC 8.2% (ERP 4.33%)        0.0561    0.0460  0.0544     0.0581  last year
terminal growth 1.5%         0.0505    0.0422  0.0491     0.0526  last year
terminal growth 3.5%         0.0564    0.0462  0.0534     0.0584  last year
tax 21%                      0.0533    0.0440  0.0518     0.0553  last year
ROIC 15%                     0.0417    0.0367  0.0407     0.0440  last year
ROIC 25%                     0.0594    0.0483  0.0553     0.0615  last year
no fade (hold 5 years)       0.1029    0.0850  0.0960     0.1032  last year

winner the same in every setting: True


Changing the fixed assumptions moves every forecast's error up or down together. The size of
the gap changes but the winner doesn't. We should rerun this once the model's forecasts are in,
and if the winner flips under any reasonable setting we'd report that.

## Part 5: Notes for the team

Person 3: save your forecasts as `data/lm_forecasts.csv` in the format from Part 3 and rerun
this notebook, with one label per run (e.g. `AI full`, `AI no MD&A`). Please also write down the
model's training cutoff. FY2025 results came out in early 2026, so a model trained after about
January 2026 may have already seen the answers.

Person 4: `output/baseline_forecasts.csv` has every forecast plus the actuals, one row per firm.
`output/dcf_valuations.csv` has every valuation and error, one row per firm per scenario; its
`ev_perfect_foresight` column is our DCF fed actual FY2025 results, not a market value.
Accuracy should only be scored on FY2025. It'd be good to report last year next to the
three-year average, split by `backlog`, and show results with and without the four flagged
firms.